In [84]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [95]:
train_df = pd.read_csv("dataset/raw/train.csv")
test_df  = pd.read_csv("dataset/raw/test.csv")

In [86]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 764271 entries, 0 to 764270
Data columns (total 28 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   DR_NO           764271 non-null  int64  
 1   Date Rptd       764271 non-null  object 
 2   DATE OCC        764271 non-null  object 
 3   TIME OCC        764271 non-null  int64  
 4   AREA            764271 non-null  int64  
 5   AREA NAME       764271 non-null  object 
 6   Rpt Dist No     764271 non-null  int64  
 7   Part 1-2        764271 non-null  int64  
 8   Crm Cd          764271 non-null  int64  
 9   Crm Cd Desc     764271 non-null  object 
 10  Mocodes         654874 non-null  object 
 11  Vict Age        764271 non-null  int64  
 12  Vict Sex        660223 non-null  object 
 13  Vict Descent    660214 non-null  object 
 14  Premis Cd       764261 non-null  float64
 15  Premis Desc     763787 non-null  object 
 16  Weapon Used Cd  259945 non-null  float64
 17  Weapon Des

In [87]:
train_df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,212018647,12/24/2021 12:00:00 AM,12/24/2021 12:00:00 AM,1115,20,Olympic,2019,1,510,VEHICLE - STOLEN,...,IC,Invest Cont,510.0,NaN,NaN,NaN,100 S VERMONT AV,NaN,34.0721,-118.2917
1,201107812,03/28/2020 12:00:00 AM,03/28/2020 12:00:00 AM,2055,11,Northeast,1149,1,210,ROBBERY,...,IC,Invest Cont,210.0,NaN,NaN,NaN,5100 N FIGUEROA ST,NaN,34.1056,-118.2008
2,240708269,04/20/2024 12:00:00 AM,04/19/2024 12:00:00 AM,1900,7,Wilshire,734,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),...,IC,Invest Cont,745.0,NaN,NaN,NaN,400 S CURSON AV,NaN,34.0675,-118.3536
3,210612806,07/16/2021 12:00:00 AM,07/15/2021 12:00:00 AM,2030,6,Hollywood,659,1,330,BURGLARY FROM VEHICLE,...,IC,Invest Cont,330.0,NaN,NaN,NaN,1300 N KINGSLEY DR,NaN,34.0951,-118.3031
4,211410554,04/27/2021 12:00:00 AM,04/27/2021 12:00:00 AM,1145,14,Pacific,1414,2,901,VIOLATION OF RESTRAINING ORDER,...,AO,Adult Other,901.0,NaN,NaN,NaN,600 SAN JUAN AV,NaN,33.9932,-118.4671


In [88]:
train_df.isnull().sum()

DR_NO                  0
Date Rptd              0
DATE OCC               0
TIME OCC               0
AREA                   0
AREA NAME              0
Rpt Dist No            0
Part 1-2               0
Crm Cd                 0
Crm Cd Desc            0
Mocodes           109397
Vict Age               0
Vict Sex          104048
Vict Descent      104057
Premis Cd             10
Premis Desc          484
Weapon Used Cd    504326
Weapon Desc       504326
Status                 1
Status Desc            0
Crm Cd 1               9
Crm Cd 2          709523
Crm Cd 3          762433
Crm Cd 4          764216
LOCATION               0
Cross Street      645010
LAT                    0
LON                    0
dtype: int64

In [96]:
def show_unique_values(df, columns=None, max_display=50):
    """
    Hiển thị các giá trị unique của các cột trong DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame cần phân tích
    columns : list, optional
        Danh sách các cột cần xem. Nếu None, sẽ hiển thị tất cả các cột
    max_display : int, default=50
        Số lượng unique values tối đa hiển thị cho mỗi cột
        
    Returns:
    --------
    dict
        Dictionary chứa unique values của từng cột
    """
    # Nếu không chỉ định columns, lấy tất cả
    if columns is None:
        columns = df.columns.tolist()
    
    # Dictionary để lưu kết quả
    unique_dict = {}
    
    print("=" * 100)
    print("UNIQUE VALUES ANALYSIS")
    print("=" * 100)
    
    for col in columns:
        if col not in df.columns:
            print(f"Cột '{col}' không tồn tại trong DataFrame")
            continue
        
        unique_vals = df[col].unique()
        n_unique = len(unique_vals)
        
        # Lấy value counts
        value_counts = df[col].value_counts(dropna=False)
        
        # Lưu vào dictionary
        unique_dict[col] = unique_vals
        
        # In thông tin
        print(f"\nFeature: {col}")
        print(f"   • Data type: {df[col].dtype}")
        print(f"   • Total unique values: {n_unique}")
        print(f"   • Missing values: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.2f}%)")
        
        # Hiển thị unique values
        if n_unique <= max_display:
            print(f"   • Unique values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")
            print(f"\n   Value Counts:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        else:
            print(f"   • Too many unique values ({n_unique}). Showing top 20:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        
        print("-" * 100)
    
    return unique_dict

In [97]:
# Ví dụ 3: Xem unique values của tất cả các cột (cẩn thận với dataset lớn)
unique_all = show_unique_values(train_df)

UNIQUE VALUES ANALYSIS

Feature: DR_NO
   • Data type: int64
   • Total unique values: 764271
   • Missing values: 0 (0.00%)
   • Too many unique values (764271). Showing top 20:
      201219926                      :      1 ( 0.00%)
      212018647                      :      1 ( 0.00%)
      201107812                      :      1 ( 0.00%)
      240708269                      :      1 ( 0.00%)
      210612806                      :      1 ( 0.00%)
      211410554                      :      1 ( 0.00%)
      210312494                      :      1 ( 0.00%)
      241007184                      :      1 ( 0.00%)
      220711619                      :      1 ( 0.00%)
      211409968                      :      1 ( 0.00%)
      221304046                      :      1 ( 0.00%)
      221415652                      :      1 ( 0.00%)
      241008091                      :      1 ( 0.00%)
      200122828                      :      1 ( 0.00%)
      230715836                      :      1 ( 0.0

## 1. Basic Processing

### 1.1. Convert Float to Int

In [98]:
def convert_float_to_int(df, column_name):
    """
    Convert a column from float to Int64 (nullable integer)
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the column to be converted
    column_name : str
        The name of the column to be converted
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with the converted column
    """
    # Check if the column exists
    if column_name not in df.columns:
        print(f"Column '{column_name}' does not exist in the DataFrame")
        return df
    
    # Check the current data type
    current_dtype = df[column_name].dtype
    print('\n')
    print(f"Current data type of '{column_name}': {current_dtype}")
    
    # Check the number of missing values
    null_count = df[column_name].isnull().sum()
    
    if null_count > 0:
        print(f"There are {null_count} null values in the column '{column_name}'")
    
    # Convert to Int64 (supports null)
    df[column_name] = df[column_name].astype('Int64')
    print(f"Converted '{column_name}' to type: {df[column_name].dtype}")
    
    return df


In [99]:
# Train set

all_train_float_cols = train_df.select_dtypes(include=['float64']).columns.tolist()
columns_change_type_train = [col for col in all_train_float_cols if col not in ['LAT', 'LON']]

for col in columns_change_type_train:
    convert_float_to_int(train_df, col)



Current data type of 'Premis Cd': float64
There are 10 null values in the column 'Premis Cd'
Converted 'Premis Cd' to type: Int64


Current data type of 'Weapon Used Cd': float64
There are 504326 null values in the column 'Weapon Used Cd'
Converted 'Weapon Used Cd' to type: Int64


Current data type of 'Crm Cd 1': float64
There are 9 null values in the column 'Crm Cd 1'
Converted 'Crm Cd 1' to type: Int64


Current data type of 'Crm Cd 2': float64
There are 709523 null values in the column 'Crm Cd 2'
Converted 'Crm Cd 2' to type: Int64


Current data type of 'Crm Cd 3': float64
There are 762433 null values in the column 'Crm Cd 3'
Converted 'Crm Cd 3' to type: Int64


Current data type of 'Crm Cd 4': float64
There are 764216 null values in the column 'Crm Cd 4'
Converted 'Crm Cd 4' to type: Int64
Converted 'Crm Cd 2' to type: Int64


Current data type of 'Crm Cd 3': float64
There are 762433 null values in the column 'Crm Cd 3'
Converted 'Crm Cd 3' to type: Int64


Current data type o

In [100]:
# Test set

all_test_float_cols = test_df.select_dtypes(include=['float64']).columns.tolist()
columns_change_type_test = [col for col in all_test_float_cols if col not in ['LAT', 'LON']]

for col in columns_change_type_test:
    convert_float_to_int(test_df, col)



Current data type of 'Premis Cd': float64
There are 2 null values in the column 'Premis Cd'
Converted 'Premis Cd' to type: Int64


Current data type of 'Weapon Used Cd': float64
There are 125994 null values in the column 'Weapon Used Cd'
Converted 'Weapon Used Cd' to type: Int64


Current data type of 'Crm Cd 1': float64
There are 2 null values in the column 'Crm Cd 1'
Converted 'Crm Cd 1' to type: Int64


Current data type of 'Crm Cd 2': float64
There are 177350 null values in the column 'Crm Cd 2'
Converted 'Crm Cd 2' to type: Int64


Current data type of 'Crm Cd 3': float64
There are 190612 null values in the column 'Crm Cd 3'
Converted 'Crm Cd 3' to type: Int64


Current data type of 'Crm Cd 4': float64
There are 191059 null values in the column 'Crm Cd 4'
Converted 'Crm Cd 4' to type: Int64


### 1.2. Convert Datetime

In [101]:
train_df['Date Rptd'] = pd.to_datetime(train_df['Date Rptd'], errors='coerce')
train_df['DATE OCC'] = pd.to_datetime(train_df['DATE OCC'], errors='coerce')

C:\Users\admin\AppData\Local\Temp\ipykernel_3832\4159458271.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_df['Date Rptd'] = pd.to_datetime(train_df['Date Rptd'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_3832\4159458271.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_df['DATE OCC'] = pd.to_datetime(train_df['DATE OCC'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_3832\4159458271.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train_df['DATE OCC'] = pd.to_datetime(train_df['DATE OCC'], errors='coerce')


In [102]:
test_df['Date Rptd'] = pd.to_datetime(test_df['Date Rptd'], errors='coerce')
test_df['DATE OCC'] = pd.to_datetime(test_df['DATE OCC'], errors='coerce')

C:\Users\admin\AppData\Local\Temp\ipykernel_3832\1783060126.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_df['Date Rptd'] = pd.to_datetime(test_df['Date Rptd'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_3832\1783060126.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_df['DATE OCC'] = pd.to_datetime(test_df['DATE OCC'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_3832\1783060126.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test_df['DATE OCC'] = pd.to_datetime(test_df['DATE OCC'], errors='coerce')


### 1.3. Define X and y

In [103]:
X_train = train_df.drop("Part 1-2", axis=1)
y_train = train_df["Part 1-2"]

X_test = test_df.drop("Part 1-2", axis=1)
y_test = test_df["Part 1-2"]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (764271, 27)
y_train shape: (764271,)
X_test shape: (191068, 27)
y_test shape: (191068,)


### 1.4. Define Categorical and Numerical Variables


In [104]:
numeric_features = ["TIME OCC", "Vict Age", "LAT", "LON"]

# Keep features that are actually in X_train
numeric_features = [c for c in numeric_features if c in X_train.columns]

# Create categorical features list
categorical_ordinal = ['Status', 'Status Desc']
categorical_nominal = [c for c in X_train.columns if c not in numeric_features + categorical_ordinal]

In [105]:
print(f" Number of numerical features: {len(numeric_features)}")
print(f" List: {numeric_features}\n")

 Number of numerical features: 4
 List: ['TIME OCC', 'Vict Age', 'LAT', 'LON']



In [106]:
print(f" Number of categorical ordinal features: {len(categorical_ordinal)}")
print(f" List: {categorical_ordinal}\n")
print(f" Number of categorical nominal features: {len(categorical_nominal)}")
print(f" List: {categorical_nominal}\n")


 Number of categorical ordinal features: 2
 List: ['Status', 'Status Desc']

 Number of categorical nominal features: 21
 List: ['DR_NO', 'Date Rptd', 'DATE OCC', 'AREA', 'AREA NAME', 'Rpt Dist No', 'Crm Cd', 'Crm Cd Desc', 'Mocodes', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc', 'Weapon Used Cd', 'Weapon Desc', 'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street']



### 1.5. Remove Zero Variance Features and Duplicated Rows

In [107]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline

class VarianceThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        self.selector = VarianceThreshold(threshold=self.threshold)
    
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
        self.numeric_cols = numeric_cols
        self.selector.fit(X[numeric_cols])
        # Save retained numeric columns
        self.retained_cols = numeric_cols[self.selector.get_support()]
        # Save dropped numeric columns
        self.dropped_cols = [col for col in numeric_cols if col not in self.retained_cols]
        print("Columns removed due to zero variance:", self.dropped_cols)
        return self
    
    def transform(self, X):
        cols_to_keep = list(self.retained_cols) + list(X.columns.difference(self.numeric_cols))
        return X[cols_to_keep]


In [108]:
class ConstantAndDuplicateRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Identify constant columns
        self.constant_cols = [col for col in X.columns if X[col].nunique() == 1]
        print("Columns removed due to having only 1 unique value:", self.constant_cols)
        return self
    
    def transform(self, X):
        # Drop constant columns
        X_cleaned = X.drop(columns=self.constant_cols, errors="ignore")
        # Drop duplicate rows
        X_cleaned = X_cleaned.drop_duplicates()
        return X_cleaned

In [109]:
cleaning_pipeline = Pipeline(steps=[
    ("var_thresh", VarianceThresholdSelector(threshold=0)),
    ("const_dup", ConstantAndDuplicateRemover())

])
X_train_cleaned = cleaning_pipeline.fit_transform(X_train.copy())
print(f"Data shape after cleaning: {X_train_cleaned.shape}")
X_test_cleaned = cleaning_pipeline.transform(X_test.copy())
print(f"Data shape after cleaning: {X_test_cleaned.shape}")

Columns removed due to zero variance: []
Columns removed due to having only 1 unique value: []
Columns removed due to having only 1 unique value: []
Data shape after cleaning: (764271, 27)
Data shape after cleaning: (764271, 27)
Data shape after cleaning: (191068, 27)
Data shape after cleaning: (191068, 27)


c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


# 2. Missing processing

### 2.1. Detaild Missing Values Analysis

In [110]:
# Detailed analysis by feature groups
print("=" * 90)
print("DETAILED MISSING VALUES ANALYSIS")
print("=" * 90)

# Determine feature type
def get_feature_type(col):
    if col in numeric_features:
        return 'Numerical'
    elif col in categorical_ordinal:
        return 'Cat-Ordinal'
    elif col in categorical_nominal:
        return 'Cat-Nominal'
    else:
        return 'Other'

# Calculate statistics
missing_info = pd.DataFrame({
    'Feature': X_train_cleaned.columns,
    'Feature Type': [get_feature_type(col) for col in X_train_cleaned.columns],
    'Missing Count': X_train_cleaned.isnull().sum().values,
    'Missing %': (X_train_cleaned.isnull().sum() / len(X_train_cleaned) * 100).values,
    'Present Count': X_train_cleaned.notnull().sum().values,
    'Data Type': X_train_cleaned.dtypes.values
})

# Classify by missing level
missing_info['Missing Level'] = pd.cut(
    missing_info['Missing %'],
    bins=[-0.1, 0, 5, 15, 100],
    labels=['None', 'Low (0-5%)', 'Medium (5-15%)', 'High (>15%)']
)

# Display only features with missing values
missing_only = missing_info[missing_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nFEATURES WITH MISSING VALUES:\n")
print(f"{'Feature':<30} | {'Type':<15} | {'Missing':<12} | {'%':<8} | {'Level':<15}")
print("-" * 95)
for idx, row in missing_only.iterrows():
    print(f"{row['Feature']:<30} | {row['Feature Type']:<15} | {row['Missing Count']:>10,} | {row['Missing %']:>6.2f}% | {row['Missing Level']:<15}")

# Overall statistics
print("\n" + "=" * 90)
print("OVERALL STATISTICS:")
print(f"   • Total observations: {len(X_train_cleaned):,}")
print(f"   • Total features: {len(X_train_cleaned.columns)}")
print(f"   • Features with missing: {len(missing_only)}")
print(f"   • Features without missing: {len(X_train_cleaned.columns) - len(missing_only)}")
print(f"   • Total missing cells: {X_train_cleaned.isnull().sum().sum():,}")
print(f"   • Overall missing rate: {(X_train_cleaned.isnull().sum().sum() / (len(X_train_cleaned) * len(X_train_cleaned.columns)) * 100):.2f}%")
print("=" * 90)

DETAILED MISSING VALUES ANALYSIS

FEATURES WITH MISSING VALUES:

Feature                        | Type            | Missing      | %        | Level          
-----------------------------------------------------------------------------------------------
Crm Cd 4                       | Cat-Nominal     |    764,216 |  99.99% | High (>15%)    
Crm Cd 3                       | Cat-Nominal     |    762,433 |  99.76% | High (>15%)    
Crm Cd 2                       | Cat-Nominal     |    709,523 |  92.84% | High (>15%)    
Cross Street                   | Cat-Nominal     |    645,010 |  84.40% | High (>15%)    
Weapon Used Cd                 | Cat-Nominal     |    504,326 |  65.99% | High (>15%)    
Weapon Desc                    | Cat-Nominal     |    504,326 |  65.99% | High (>15%)    
Mocodes                        | Cat-Nominal     |    109,397 |  14.31% | Medium (5-15%) 
Vict Descent                   | Cat-Nominal     |    104,057 |  13.62% | Medium (5-15%) 
Vict Sex                  

### 2.2. Feature Dropping Strategy

#### a. High Missing Values (Data Quality Issue)
- **Crm Cd 2, 3, 4**: >99% missing - too sparse to impute
- **Cross Street**: Excessive missing values

#### b. Data Leakage (Direct Target Indicators)
**Weapon Used Cd**, **Weapon Desc**, and **Mocodes** will be dropped because:
- **Leakage Risk**: These features directly indicate crime severity (Part 1 vs Part 2)
- **Circular Logic**: 
  - Crimes with weapons = Part 1, no weapons = Part 2
  - Mocodes (Modus Operandi codes) describe crime methods/severity
- **Real-world**: Using these to predict severity is restating the definition, not prediction

#### c. High Target Correlation
**Crm Cd 1** will be dropped due to:
- Strong correlation with target variable
- Crime codes are assigned based on severity, which determines Part 1-2 classification
- Causes overfitting; primary `Crm Cd` already captures crime type

#### d. No Practical Predictive Value
**LAT** and **LON** (latitude/longitude coordinates) will be dropped because:
- Geographic coordinates alone cannot explain crime severity
- Location doesn't determine whether a crime is Part 1 or Part 2
- Area/district information is already captured by other location features
- Exact coordinates add noise without meaningful signal for severity classification

| Feature | Reason |
|---------|--------|
| Crm Cd 2, 3, 4, Cross Street | >99% missing |
| Weapon Used Cd, Weapon Desc, Mocodes | Data leakage |
| Crm Cd 1 | High target correlation |
| LAT, LON | No practical predictive value |

In [116]:
# Drop features: high missing + data leakage + high correlation
all_cols_to_drop = ['Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 
                    'Cross Street', 'Weapon Used Cd', 'Weapon Desc', 'Mocodes', 'LAT', 'LON']

# Check existence
cols_to_drop = [col for col in all_cols_to_drop if col in X_train_cleaned.columns]

print(f"Dropping {len(cols_to_drop)} features:")
for col in cols_to_drop:
    if col in X_train_cleaned.columns:
        missing_pct = (X_train_cleaned[col].isnull().sum() / len(X_train_cleaned)) * 100
        reason = "High missing" if missing_pct > 50 else "Data leakage / High correlation"
        print(f"   • {col}: {reason}")

# Drop from both sets
X_train_cleaned = X_train_cleaned.drop(columns=cols_to_drop, errors='ignore')
X_test_cleaned = X_test_cleaned.drop(columns=cols_to_drop, errors='ignore')

print(f"\nShape after dropping: Train {X_train_cleaned.shape}, Test {X_test_cleaned.shape}")

Dropping 1 features:
   • Mocodes: Data leakage / High correlation

Shape after dropping: Train (764271, 17), Test (191068, 17)


In [114]:
# Check remaining missing values after dropping
print("NUMERIC FEATURES WITH MISSING:")
for col in numeric_features:
    if col in X_train_cleaned.columns and X_train_cleaned[col].isnull().sum() > 0:
        print(f"   • {col}")

print("\nCATEGORICAL FEATURES WITH MISSING:")
all_categorical = categorical_ordinal + categorical_nominal
for col in all_categorical:
    if col in X_train_cleaned.columns and X_train_cleaned[col].isnull().sum() > 0:
        print(f"   • {col}")

NUMERIC FEATURES WITH MISSING:

CATEGORICAL FEATURES WITH MISSING:
   • Status
   • Mocodes
   • Vict Sex
   • Vict Descent
   • Premis Cd
   • Premis Desc


### 2.3. Create Pipeline


In [ ]:
# Categorical Nominal
nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

In [ ]:
# Step 4: Create ColumnTransformer
# preprocessor = ColumnTransformer(
#  transformers=[
#       ('num_simple', numeric_simple_pipeline, numeric_simple),
#       ('num_knn', numeric_knn_pipeline, numeric_knn),
#       ('cat_nominal', nominal_pipeline, categorical_nominal)
#   ],
#   remainder='passthrough', 
#   verbose_feature_names_out=False
# )

In [ ]:
# Step 5: Fit and transform
# X_train_processed = preprocessor.fit_transform(X_train)
# X_test_processed = preprocessor.transform(X_test)

# print(f"\nRESULTS:")
# print(f"   • Original X_train shape: {X_train.shape}")
# print(f"   • Processed X_train shape: {X_train_processed.shape}")
# print(f"   • Processed X_test shape: {X_test_processed.shape}")
# print(f"   • Missing values after: {np.isnan(X_train_processed).sum()}")

KeyboardInterrupt: 

In [ ]:
# Step 6: Get feature names (optional)

# feature_names = preprocessor.get_feature_names_out()

# print(f"\nFEATURE SUMMARY:")
# print(f"   • Total features: {len(feature_names)}")
# print(f"   • Numerical (simple): {len(numeric_simple)}")
# print(f"   • Numerical (KNN): {len(numeric_knn)}")
# print(f"   • Ordinal: {len(categorical_ordinal)}")
# print(f"   • Nominal (one-hot): {len(categorical_nominal)}")

# Convert to DataFrame
# X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
# X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

## Outlier Processing

In [101]:
outlier_cols = [
    'Age', 'Annual Income', 'Number of Dependents',
    'Health Score', 'Previous Claims', 'Vehicle Age', 
    'Credit Score', 'Insurance Duration', 'Premium Amount'
]

print("=" * 90)
print("PART 1: OUTLIER ANALYSIS (IQR METHOD)")
print("=" * 90)

# Check which columns actually exist in the dataframe
available_cols = [c for c in outlier_cols if c in X_train.columns]
report_data = []

for col in available_cols:
    # 1. Calculate IQR
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Define Limits (for reporting only, not applied yet)
    lower_bound = max(Q1 - 1.5 * IQR, 0)
    upper_bound = Q3 + 1.5 * IQR
    
    # 3. Count actual outliers
    outliers_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)
    num_outliers = outliers_mask.sum()
    pct_outliers = (num_outliers / len(X_train)) * 100
    
    if num_outliers > 0:
        report_data.append({
            'Feature': col,
            'Count': num_outliers,
            'Percent': f"{pct_outliers:.2f}%",
            'Rec. Lower': f"{lower_bound:.2f}", # Recommended Lower Limit
            'Rec. Upper': f"{upper_bound:.2f}", # Recommended Upper Limit
            'Min': f"{X_train[col].min():.2f}",
            'Max': f"{X_train[col].max():.2f}"
        })

# 4. Display Report
if len(report_data) > 0:
    report_df = pd.DataFrame(report_data)
    print(f"Outliers detected in {len(report_data)} columns:\n")
    # Print formatted table
    print(report_df.to_string(index=False, justify='left'))
else:
    print("No significant outliers found using the IQR method.")

# --- OVERALL STATISTICS SECTION ---
total_outlier_cells = sum(item['Count'] for item in report_data)
total_features_analyzed = len(available_cols)
total_cells_analyzed = len(X_train) * total_features_analyzed

print("\n" + "=" * 90)
print("OVERALL OUTLIER STATISTICS:")
print(f"   • Total observations: {len(X_train):,}")
print(f"   • Features analyzed: {total_features_analyzed} (Numeric columns only)")
print(f"   • Features with outliers: {len(report_data)}")
print(f"   • Total outlier cells: {total_outlier_cells:,}")
if total_cells_analyzed > 0:
    print(f"   • Overall outlier rate: {(total_outlier_cells / total_cells_analyzed * 100):.2f}%")
else:
    print(f"   • Overall outlier rate: 0.00%")
print("=" * 90)

PART 1: OUTLIER ANALYSIS (IQR METHOD)
Outliers detected in 2 columns:

Feature          Count Percent Rec. Lower Rec. Upper Min  Max      
  Annual Income 67132  5.59%   0.00       99583.50   1.00 149997.00
Previous Claims 62066  5.17%   0.00           2.50   0.00      9.00

OVERALL OUTLIER STATISTICS:
   • Total observations: 1,200,000
   • Features analyzed: 8 (Numeric columns only)
   • Features with outliers: 2
   • Total outlier cells: 129,198
   • Overall outlier rate: 1.35%


In [ ]:
print("=" * 90)
print("PART 2: OUTLIER PROCESSING (APPLYING CAPPING)")
print("=" * 90)

processed_count = 0

for col in available_cols:
    # Recalculate limits (ensures consistency)
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = max(Q1 - 1.5 * IQR, 0)
    upper_bound = Q3 + 1.5 * IQR
    
    # Check for outliers before processing
    outliers_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)
    
    if outliers_mask.sum() > 0:
        # --- APPLY CAPPING ---
        # np.clip forces values < lower to lower and > upper to upper
        X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
        print(f"✓ Processed column: {col:<25} (Limits: {lower_bound:.2f} - {upper_bound:.2f})")
        processed_count += 1

if processed_count == 0:
    print("No columns required processing.")
else:
    print("-" * 90)
    print(f"Done! Applied capping to {processed_count} columns.")

## Categorical Encoder

In [ ]:
ordinal_features = ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']
nominal_features = [col for col in categorical_features if col not in ordinal_features]

ordinal_education = ["High School", "Bachelor's", "Master's", "PhD"]
ordinal_policy = ["Basic", "Comprehensive", "Premium"]
ordinal_feedback = ["Poor", "Average", "Good"]
ordinal_exercise = ["Rarely", "Monthly", "Weekly", "Daily"]

In [ ]:
# ColumnTransformer để xử lý theo nhóm
preprocessor = ColumnTransformer(
    transformers=[
        # OneHot cho nominal categorical - ignore unknown values
        ("encode_cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),   
        # Ordinal Encoder - use sentinel value for unknown
        ("encode_ord", OrdinalEncoder(categories=[ordinal_education, ordinal_policy, ordinal_feedback, ordinal_exercise],
             handle_unknown='use_encoded_value',
             unknown_value=-1 
         ), 
         ordinal_features)
    ],
    remainder="passthrough"
)

X_transformed = preprocessor.fit_transform(X_train, y_train)

In [ ]:
# Lấy tên các cột sau khi transform
feature_names = preprocessor.get_feature_names_out()

print("=" * 90)
print("TRANSFORMED FEATURE NAMES")
print("=" * 90)
print(f"Total features after transformation: {len(feature_names)}\n")

# Convert transformed data to DataFrame
X_train_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X_train.index)
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

# Add target variable back to train data
train_processed = X_train_transformed.copy()
train_processed['Premium Amount'] = y_train

print(f"✓ Train data shape: {train_processed.shape}")
print(f"✓ Test data shape: {X_test_transformed.shape}")
print(f"\nFeature names preview:")
for i, name in enumerate(feature_names[:10], 1):
    print(f"   {i}. {name}")
if len(feature_names) > 10:
    print(f"   ... and {len(feature_names) - 10} more features")
print("=" * 90)

In [ ]:
# Save processed data to CSV
import os

# Create processed directory if not exists
os.makedirs("dataset/processed", exist_ok=True)

# Save train data (with target variable)
train_processed.to_csv("dataset/processed/train_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/train_processed.csv ({train_processed.shape})")

# Save test data (without target variable)
X_test_transformed.to_csv("dataset/processed/test_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/test_processed.csv ({X_test_transformed.shape})")

print("\n" + "=" * 90)
print("ALL FILES SAVED SUCCESSFULLY!")
print(f"   • Train: {train_processed.shape[0]:,} rows × {train_processed.shape[1]} columns")
print(f"   • Test:  {X_test_transformed.shape[0]:,} rows × {X_test_transformed.shape[1]} columns")
print("=" * 90)